# Zebra-Puzzle GRPO Fine-tune (Gemma 4 E2B)

Copyright (c) 2026 Thinkube Contributors. Licensed under Apache-2.0.
SPDX-License-Identifier: Apache-2.0

This notebook uses the Unsloth library (Apache-2.0) and the TRL library (Apache-2.0).
Training data generated by the included zebra_dataset.py module (Apache-2.0).
Out-of-distribution evaluation uses AllenAI's ZebraLogicBench dataset
(https://huggingface.co/datasets/allenai/ZebraLogicBench).
This notebook is an independent work and is NOT derived from any LGPL-3.0
notebook in the unslothai/notebooks repository.

---

## What This Notebook Does

Fine-tunes **Gemma 4 E2B** on zebra logic puzzles using **GRPO** (Group Relative Policy Optimization).
Evaluates on two sets to distinguish format-fitting from real reasoning improvement:

| Eval Set | Source | What It Measures |
|----------|--------|------------------|
| **In-distribution** | Same generator, different seed | Pure learning on our format |
| **ZebraLogicBench** | AllenAI benchmark (different prompt/clue style) | Transfer — did the model learn to reason? |

**Target hardware:** NVIDIA DGX Spark (Blackwell, unified memory, ~9 GB for E2B GRPO)

## [Optional] Install Dependencies

Skip if running in Unsloth's container or thinkube's `fine-tuning` venv.

In [ ]:
# Uncomment to install:
# !pip install unsloth trl datasets python-constraint matplotlib

## Imports

In [ ]:
import json
import random
import re
import time

import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset, load_dataset

import zebra_dataset

## Configuration

All training-relevant numbers live here. Set `SMOKE_TEST = True` for a quick
pipeline check (~2 min) before committing to a full run.

In [ ]:
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LEN = 2048
LORA_RANK = 16
LORA_ALPHA = 16

NUM_TRAIN_PUZZLES = 2000
NUM_EVAL_PUZZLES = 200
TRAIN_SEED = 1
EVAL_SEED = 99
SEED = 3407

NUM_GENERATIONS = 8
LEARNING_RATE = 5e-6
MAX_STEPS = 250
GRADIENT_ACCUMULATION = 4
KL_BETA = 0.04
MAX_PROMPT_LENGTH = 768
MAX_COMPLETION_LENGTH = 1024

OUTPUT_DIR = "./zebra_grpo_e2b"

SMOKE_TEST = False

if SMOKE_TEST:
    NUM_TRAIN_PUZZLES = 100
    NUM_EVAL_PUZZLES = 50
    MAX_STEPS = 10
    print("SMOKE TEST mode: reduced dataset and steps")

print(f"Model: {MODEL_NAME}")
print(f"Train: {NUM_TRAIN_PUZZLES}, Eval: {NUM_EVAL_PUZZLES}")
print(f"LoRA rank: {LORA_RANK}, LR: {LEARNING_RATE}, Steps: {MAX_STEPS}")

## Load Model & Tokenizer

bf16 precision (required by Gemma 4), `fast_inference=False` (GRPO needs standard generation path).

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    dtype=torch.bfloat16,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.2f}%)")

## Generate Training & In-Distribution Eval Data

Disjoint seeds ensure no overlap between training and evaluation puzzles.

In [ ]:
def make_puzzles(num_puzzles, seed):
    """Generate puzzles with a fixed RNG."""
    rng = random.Random(seed)
    puzzles = []
    while len(puzzles) < num_puzzles:
        theme = rng.choice(zebra_dataset.THEMES)
        puzzle = zebra_dataset.generate(theme, 5, rng)
        if puzzle is not None:
            puzzles.append(puzzle)
        if len(puzzles) % 200 == 0 and len(puzzles) > 0:
            print(f"  {len(puzzles)}/{num_puzzles}...")
    return puzzles

print("Generating training puzzles...")
train_puzzles = make_puzzles(NUM_TRAIN_PUZZLES, TRAIN_SEED)

print("Generating evaluation puzzles...")
eval_puzzles = make_puzzles(NUM_EVAL_PUZZLES, EVAL_SEED)

# Disjointness assertion
train_sigs = {json.dumps(p["solution"], sort_keys=True) for p in train_puzzles}
eval_sigs = {json.dumps(p["solution"], sort_keys=True) for p in eval_puzzles}
assert train_sigs.isdisjoint(eval_sigs), "train/eval overlap detected — check seeds"
print(f"Train/eval disjoint: confirmed ({len(train_sigs)} vs {len(eval_sigs)} unique)")

## Load ZebraLogicBench

AllenAI's public benchmark for zebra-puzzle reasoning. We filter to 5x5 puzzles
for the headline number and convert to our internal format for scoring.

In [ ]:
zlogic = load_dataset("allenai/ZebraLogicBench", split="test")
print(f"ZebraLogicBench: {len(zlogic)} puzzles")
print(f"Schema: {zlogic.features}")
print(f"\nSample row keys: {list(zlogic[0].keys())}")
print(f"Sample size: {zlogic[0].get('size', 'N/A')}")

In [ ]:
def zlogic_to_internal(row):
    """Convert a ZebraLogicBench row to our puzzle dict shape.

    Returns (puzzle_dict, original_prompt) or (None, None) if conversion fails.
    The puzzle_dict has the shape zebra_dataset.reward() expects.
    """
    try:
        # Parse size (e.g. "5x5" -> N=5 houses, M=5 categories)
        size_str = row.get("size", "")
        if "x" in size_str:
            parts = size_str.split("x")
            n_houses = int(parts[0])
        else:
            return None, None

        # Parse the solution — typically a JSON string or dict
        sol_raw = row.get("solution", {})
        if isinstance(sol_raw, str):
            sol_data = json.loads(sol_raw)
        else:
            sol_data = sol_raw

        # Build solution dict: {category: [value_at_house_0, value_at_house_1, ...]}
        # ZebraLogicBench solution format may vary — adapt based on actual schema
        if isinstance(sol_data, dict):
            solution = {}
            categories = {}
            for cat, values in sol_data.items():
                if isinstance(values, list) and len(values) == n_houses:
                    solution[cat] = values
                    categories[cat] = sorted(set(values))
        elif isinstance(sol_data, list):
            # List of dicts, one per house
            all_cats = set()
            for house in sol_data:
                all_cats.update(house.keys())
            solution = {cat: [] for cat in all_cats}
            categories = {cat: set() for cat in all_cats}
            for house in sol_data:
                for cat in all_cats:
                    val = house.get(cat, "")
                    solution[cat].append(val)
                    categories[cat].add(val)
            categories = {cat: sorted(vals) for cat, vals in categories.items()}
        else:
            return None, None

        puzzle = {
            "N": n_houses,
            "categories": categories,
            "solution": solution,
            "clues": [],
        }

        # Verify round-trip: reward(format_expected(puzzle), puzzle) should be 1.25
        expected = zebra_dataset.format_expected(puzzle)
        check = zebra_dataset.reward(expected, puzzle)
        if check < 1.2:
            return None, None

        original_prompt = row.get("puzzle", "")
        return puzzle, original_prompt

    except Exception:
        return None, None


# Convert and filter to 5x5
zlogic_puzzles = []
zlogic_all_sizes = {}

for row in zlogic:
    puzzle, orig_prompt = zlogic_to_internal(row)
    if puzzle is None:
        continue
    size = row.get("size", "?")
    zlogic_all_sizes.setdefault(size, []).append((puzzle, orig_prompt))
    if puzzle["N"] == 5:
        zlogic_puzzles.append(puzzle)

print(f"\nZebraLogicBench conversion:")
for size, items in sorted(zlogic_all_sizes.items()):
    print(f"  {size}: {len(items)} puzzles")
print(f"\n5x5 puzzles for evaluation: {len(zlogic_puzzles)}")

# Verify a few round-trips
for i in range(min(3, len(zlogic_puzzles))):
    p = zlogic_puzzles[i]
    check = zebra_dataset.reward(zebra_dataset.format_expected(p), p)
    print(f"  Round-trip check puzzle {i}: reward={check:.2f} (should be 1.25)")

## System Prompt

In [ ]:
SYSTEM_PROMPT = (
    "You are a logic puzzle solver. Reason step by step through the clues, "
    "then write the final answer with exactly one line per house in this format:\n"
    "House 1: category1=value | category2=value | ...\n"
    "House 2: category1=value | category2=value | ...\n"
    "(and so on for all houses)"
)

# Prepare training dataset as chat format
def puzzle_to_chat(puzzle):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": zebra_dataset.format_prompt(puzzle)},
        ],
        "puzzle_solution": json.dumps(puzzle["solution"]),
        "puzzle_categories": json.dumps(list(puzzle["categories"].keys())),
        "puzzle_n": puzzle["N"],
    }

train_dataset = Dataset.from_list([puzzle_to_chat(p) for p in train_puzzles])
print(f"Training dataset: {len(train_dataset)} rows")
print(f"System prompt tokens: ~{len(SYSTEM_PROMPT.split())} words")

## Baseline Evaluation

Evaluate the untrained model on both eval sets. Greedy (temperature=0) for headline numbers.

In [ ]:
def evaluate_model(model, tokenizer, puzzles, temperature=0.0, max_new_tokens=MAX_COMPLETION_LENGTH):
    """Evaluate model on puzzles. Returns (rewards, samples, cell_accs)."""
    FastLanguageModel.for_inference(model)
    rewards, cell_accs, samples = [], [], []

    for idx, puzzle in enumerate(puzzles):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": zebra_dataset.format_prompt(puzzle)},
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        gen_kwargs = dict(max_new_tokens=max_new_tokens)
        if temperature > 0:
            gen_kwargs.update(temperature=temperature, do_sample=True)
        else:
            gen_kwargs.update(do_sample=False)

        with torch.no_grad():
            output_ids = model.generate(**inputs, **gen_kwargs)

        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        score = zebra_dataset.reward(generated, puzzle)
        # Cell accuracy = score without perfect bonus, capped at 1.0
        cell_acc = min(score, 1.0)
        rewards.append(score)
        cell_accs.append(cell_acc)

        if len(samples) < 2:
            samples.append({
                "idx": idx, "reward": score,
                "expected": zebra_dataset.format_expected(puzzle),
                "generated": generated[:800],
            })

        if (idx + 1) % 50 == 0:
            print(f"  {idx+1}/{len(puzzles)}, cell_acc={sum(cell_accs)/len(cell_accs):.3f}")

    FastLanguageModel.for_training(model)
    puzzle_acc = sum(1 for r in rewards if r >= 1.25) / len(rewards)
    mean_cell = sum(cell_accs) / len(cell_accs)
    return {"puzzle_acc": puzzle_acc, "cell_acc": mean_cell, "rewards": rewards, "samples": samples}


# Quick sanity check
print("Sanity check on 10 eval puzzles...")
quick = evaluate_model(model, tokenizer, eval_puzzles[:10])
print(f"Quick baseline cell_acc: {quick['cell_acc']:.3f}")
if quick["cell_acc"] >= 0.45:
    print("WARNING: Baseline already strong (>= 0.45). Consider harder puzzles.")

In [ ]:
print("Baseline: in-distribution eval...")
baseline_indist = evaluate_model(model, tokenizer, eval_puzzles)

print("\nBaseline: ZebraLogicBench 5x5...")
baseline_zlogic = evaluate_model(model, tokenizer, zlogic_puzzles) if zlogic_puzzles else None

print(f"\nBaseline Results:")
print(f"  In-dist  puzzle_acc={100*baseline_indist['puzzle_acc']:.1f}%  cell_acc={baseline_indist['cell_acc']:.3f}")
if baseline_zlogic:
    print(f"  ZebraLogic  puzzle_acc={100*baseline_zlogic['puzzle_acc']:.1f}%  cell_acc={baseline_zlogic['cell_acc']:.3f}")

print("\n--- Sample (in-dist) ---")
for s in baseline_indist["samples"][:1]:
    print(f"Puzzle {s['idx']} (reward={s['reward']:.3f}):")
    print(f"Expected:\n{s['expected']}")
    print(f"Generated:\n{s['generated'][:400]}")

## Reward Wrapper for TRL

Format bonus (+0.05) when the output has exactly N `House N:` lines — accelerates format learning.

In [ ]:
def zebra_reward_func(completions, puzzle_solution, puzzle_categories, puzzle_n, **kwargs):
    """GRPO reward: zebra_dataset.reward() + format bonus."""
    rewards = []
    for completion, sol_json, cats_json, n_val in zip(
        completions, puzzle_solution, puzzle_categories, puzzle_n
    ):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        n = int(n_val)

        puzzle = {
            "solution": json.loads(sol_json),
            "categories": {c: [] for c in json.loads(cats_json)},
            "N": n,
        }
        score = zebra_dataset.reward(text, puzzle)

        # Format bonus: correct number of House N: lines
        house_lines = re.findall(r"^\s*House\s+\d+:", text, re.MULTILINE)
        if len(house_lines) == n:
            score += 0.05

        rewards.append(float(score))
    return rewards

print("Reward function ready.")

## GRPO Trainer Setup

In [ ]:
training_config = GRPOConfig(
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS,
    max_grad_norm=0.1,
    beta=KL_BETA,
    logging_steps=1,
    save_steps=MAX_STEPS,
    report_to="none",
    output_dir=OUTPUT_DIR,
    seed=SEED,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[zebra_reward_func],
    args=training_config,
    train_dataset=train_dataset,
)

print(f"Trainer ready: {MAX_STEPS} steps, {NUM_GENERATIONS} generations/prompt")

## Training

Rewards may not increase for 100-150 steps. Ctrl-C leaves the trainer state recoverable.

In [ ]:
start_time = time.time()

try:
    trainer.train()
except KeyboardInterrupt:
    print("\nTraining interrupted — model state is recoverable.")

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/60:.1f} minutes")

try:
    import matplotlib.pyplot as plt
    losses = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
    if losses:
        steps, vals = zip(*losses)
        plt.figure(figsize=(10, 4))
        plt.plot(steps, vals, alpha=0.7)
        plt.xlabel("Step"); plt.ylabel("Loss")
        plt.title("GRPO Training Loss")
        plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    losses = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
    for step, loss in losses[-10:]:
        print(f"  step {step}: {loss:.4f}")

## Post-Training Evaluation

Re-evaluate on both eval sets. Greedy decoding for headline numbers.

In [ ]:
print("Trained: in-distribution eval...")
trained_indist = evaluate_model(model, tokenizer, eval_puzzles)

print("\nTrained: ZebraLogicBench 5x5...")
trained_zlogic = evaluate_model(model, tokenizer, zlogic_puzzles) if zlogic_puzzles else None

# Comparison table
print(f"\n{'':24s} {'baseline':>10s} {'trained':>10s} {'delta':>10s}")
print(f"{'='*56}")

d_ip = trained_indist['puzzle_acc'] - baseline_indist['puzzle_acc']
d_ic = trained_indist['cell_acc'] - baseline_indist['cell_acc']
print(f"{'in-dist  puzzle acc':24s} {100*baseline_indist['puzzle_acc']:9.1f}% {100*trained_indist['puzzle_acc']:9.1f}% {100*d_ip:+9.1f}%")
print(f"{'in-dist  cell acc':24s} {baseline_indist['cell_acc']:10.3f} {trained_indist['cell_acc']:10.3f} {d_ic:+10.3f}")

if baseline_zlogic and trained_zlogic:
    d_zp = trained_zlogic['puzzle_acc'] - baseline_zlogic['puzzle_acc']
    d_zc = trained_zlogic['cell_acc'] - baseline_zlogic['cell_acc']
    print(f"{'zlogic   puzzle acc':24s} {100*baseline_zlogic['puzzle_acc']:9.1f}% {100*trained_zlogic['puzzle_acc']:9.1f}% {100*d_zp:+9.1f}%")
    print(f"{'zlogic   cell acc':24s} {baseline_zlogic['cell_acc']:10.3f} {trained_zlogic['cell_acc']:10.3f} {d_zc:+10.3f}")

print()
if d_ic >= 0.15:
    print(f"In-dist target MET: +{d_ic:.3f} >= +0.15")
else:
    print(f"In-dist target NOT met: +{d_ic:.3f} < +0.15")

if baseline_zlogic and trained_zlogic:
    if d_zc >= 0.08:
        print(f"ZebraLogic target MET: +{d_zc:.3f} >= +0.08")
    else:
        print(f"ZebraLogic target NOT met: +{d_zc:.3f} < +0.08")

## Qualitative Comparison

Side-by-side on one in-distribution puzzle and one ZebraLogicBench puzzle.

In [ ]:
from IPython.display import display, Markdown

output = ""

# In-distribution sample
if baseline_indist["samples"] and trained_indist["samples"]:
    b, t = baseline_indist["samples"][0], trained_indist["samples"][0]
    output += f"""## In-Distribution Puzzle {b['idx']}

**Expected:**
```
{b['expected']}
```

**Base model** (reward={b['reward']:.3f}):
```
{b['generated'][:500]}
```

**Trained model** (reward={t['reward']:.3f}):
```
{t['generated'][:500]}
```

---

"""

# ZebraLogicBench sample
if baseline_zlogic and trained_zlogic and baseline_zlogic["samples"] and trained_zlogic["samples"]:
    b, t = baseline_zlogic["samples"][0], trained_zlogic["samples"][0]
    output += f"""## ZebraLogicBench Puzzle {b['idx']}

**Expected:**
```
{b['expected']}
```

**Base model** (reward={b['reward']:.3f}):
```
{b['generated'][:500]}
```

**Trained model** (reward={t['reward']:.3f}):
```
{t['generated'][:500]}
```
"""

display(Markdown(output))

## Save LoRA Adapter

In [ ]:
import os

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

abs_path = os.path.abspath(OUTPUT_DIR)
print(f"LoRA adapter saved to: {abs_path}")
print(f"Contents: {os.listdir(abs_path)}")

## [Optional] Export to GGUF

GGUF export from Unsloth is best-effort for very new model architectures.

In [ ]:
# Uncomment to export:
# model.save_pretrained_gguf("zebra_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF export complete.")

---

**Citation:**

```
ZebraLogic: Lin, B.Y., Le Bras, R., Choi, Y. (2024). ZebraLogic: Benchmarking
the Logical Reasoning Ability of Language Models.
https://huggingface.co/spaces/allenai/ZebraLogic
```